# Loan Application Approval Prediction System
This notebook builds a generalized, robust machine learning pipeline to predict loan approval decisions and approval probabilities from multi-source applicant data.

### Workflow:
1. Load clean combined dataset (`clean_loan_approval.csv`)
2. Exploratory Data Analysis & Target Distribution
3. Train/Test Split (80/20 Stratified)
4. Pipeline Preprocessing (ColumnTransformer: OneHotEncoder + SimpleImputer + StandardScaler)
5. Model Training & Comparison (Logistic Regression, Random Forest, Gradient Boosting)
6. 5-Fold Stratified Cross-Validation
7. Best Model Selection & Evaluation
8. Pipeline Serialization to `loan_model.joblib`


## 1. Environment & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)


## 2. Load Clean Dataset
Loading the standardized 22-column dataset generated by `combine_datasets.py`.

In [ ]:
df = pd.read_csv('clean_loan_approval.csv')
print("Dataset Shape:", df.shape)
df.head()


## 3. Exploratory Data Analysis & Target Distribution

In [ ]:
print("Target Counts:")
print(df['target'].value_counts())
print("
Target Percentage:")
print((df['target'].value_counts(normalize=True).round(4) * 100))

print("
Missing Values Summary:")
missing = df.isnull().sum()
print(missing[missing > 0])


## 4. Train / Test Split
Using an 80/20 stratified split to preserve class ratios.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

cat_cols = ['gender', 'married', 'education', 'self_employed', 'property_area']
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")


## 5. Preprocessing Pipeline
- **Numerical Features**: Median imputation with missing value indicators + standard scaling.
- **Categorical Features**: Missing category imputation + One-Hot Encoding.

In [ ]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])


## 6. Model Training & Comparison
Comparing:
1. **Logistic Regression** (Linear baseline, highly interpretable)
2. **Random Forest** (Nonlinear ensemble, handles interactions)
3. **Gradient Boosting** (Sequential boosting trees, high predictive power)

In [ ]:
candidate_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
comparison_records = []
fitted_pipelines = {}

for name, model in candidate_models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    
    # 5-fold Cross-Validation on training data
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    # Fit and evaluate on hold-out test set
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    comparison_records.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        '5-Fold CV ROC-AUC': cv_scores.mean()
    })

results_df = pd.DataFrame(comparison_records).set_index('Model')
results_df.round(4)


## 7. Model Evaluation & Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, pipe) in zip(axes, fitted_pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Rejected', 'Approved'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f"{name}\nAccuracy: {acc:.2%}")

plt.tight_layout()
plt.show()


## 8. Exporting the Final Pipeline to `loan_model.joblib`
Bundle the end-to-end preprocessing + best trained model into `loan_model.joblib`.

In [ ]:
best_model_name = results_df['ROC-AUC'].idxmax()
best_pipe = fitted_pipelines[best_model_name]
print(f"Selected Best Model: {best_model_name}")

bundle = {
    'pipeline': best_pipe,
    'model_name': best_model_name,
    'cat_features': cat_cols,
    'num_features': num_cols,
    'feature_order': list(X.columns),
    'metrics': results_df.loc[best_model_name].to_dict()
}

joblib.dump(bundle, 'loan_model.joblib')
print("Pipeline successfully saved to loan_model.joblib!")
